In [ ]:
# Import Spark SQL data types used to explicitly define the schema for incoming CSV data
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, TimestampType

# Import functions to build a key-value map column and to create literal (constant) values
from pyspark.sql.functions import create_map, lit

In [ ]:
# Read the job parameters passed in via base_parameters in the bundle's job.yml.
# These widgets only resolve to real values when the notebook runs as part of an
# actual job (e.g. via "databricks bundle run") — running the cell standalone
# in the notebook editor won't have these widgets set.
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")

# Injected from the bundle's job.yml — dynamically
# resolves to citibike_dev/test/prod depending on
# which target the job was deployed to
catalog = dbutils.widgets.get("catalog")

In [3]:
# Explicitly define the schema for the Bronze layer, converting raw string fields
# from Landing (started_at/ended_at, lat/lng) into proper timestamp and decimal types
schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DecimalType(9, 6), True),
    StructField("start_lng", DecimalType(9, 6), True),
    StructField("end_lat", DecimalType(9, 6), True),
    StructField("end_lng", DecimalType(9, 6), True),
    StructField("member_casual", StringType(), True)
])

In [ ]:
# Read raw CSV from the Landing volume, applying the explicit schema defined above
# instead of relying on Spark's automatic schema inference (which can misdetect types).
# Uses the dynamic "catalog" variable so this notebook works correctly across
# dev/test/prod without any code changes — only the deployed target changes
df = spark.read.csv(f"/Volumes/{catalog}/00_landing/source_citibike_data/JC-202503-citibike-tripdata.csv", header=True, schema=schema)

In [ ]:
# Add a metadata column as a map of key/value pairs for pipeline lineage/observability.
# Values are placeholders here — they'll be populated with real job/run identifiers
# once this notebook runs as part of an actual Databricks job (e.g. via widgets/parameters)
df = df.withColumn("metadata", create_map(
    lit("pipeline_id"), lit(pipeline_id),
    lit("run_id"), lit(run_id),
    lit("task_id"), lit(task_id),
    lit("processed_timestamp"), lit(processed_timestamp)
))

In [ ]:
# Write the transformed DataFrame as a managed Delta table in the Bronze schema.
# "overwrite" mode replaces existing data; "overwriteSchema" allows the table's
# schema to change too (needed since we're still iterating on the Bronze structure).
# Uses the dynamic "catalog" variable
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.01_bronze.jc_citibike")